# Код написан с помощью ChatGPT 6 Astra (Режим высокий-максимальный)

# ThermoMPNN + SWARM

Сравниваем исходную ThermoMPNN и четыре новые головы: `mlp`, `self`,
`static`, `swarm`. ThermoMPNN заморожена; новые головы обучаются с нуля.
Цель: **`-ddG_ML`, ккал/моль; положительное значение — дестабилизация**.

Официальное разбиение: 239 train / 31 validation / 28 test.
Используем 10 seeds (42–51), 40 эпох, Adam с learning rate 0.001.
Лучшие веса выбираем по validation macro MAE; test используем после этого.

`RUN_NAME` задаёт один эксперимент и соответствующие папки в `results/` и `weights/`. 
Повторный запуск загружает готовые головы;
Для нового эксперимента задайте другое имя. 
Для прежнего запуска укажите имя его папки в обеих директориях.

Нужны окружение NeuroPP, установленная ThermoMPNN и данные MegaScale
в `data/megascale/` (распакованные файлы либо уже скачанные архивы).

In [1]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import os
import pickle
import random
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from omegaconf import OmegaConf
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT
THERMOMPNN_DIR = PROJECT_ROOT / "external" / "ThermoMPNN"
DATA_DIR = PROJECT_ROOT / "data" / "megascale"
CSV_PATH = DATA_DIR / "Processed_K50_dG_datasets" / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
PDB_DIR = DATA_DIR / "AlphaFold_model_PDBs"
SPLITS_PATH = THERMOMPNN_DIR / "dataset_splits" / "mega_splits.pkl"
CHECKPOINT_PATH = THERMOMPNN_DIR / "models" / "thermoMPNN_default.pt"
CACHE_DIR = PROJECT_ROOT / "data" / "features" / "thermompnn"
DEVICE = torch.device("cpu")

RUN_NAME = "swarm_final"
RESULTS_DIR = PROJECT_ROOT / "results" / RUN_NAME
WEIGHTS_DIR = PROJECT_ROOT / "weights" / RUN_NAME
TEST_RESULTS_DIR = RESULTS_DIR / "test"
SEEDS = list(range(42, 52))
MODEL_KINDS = ["mlp", "self", "static", "swarm"]
EPOCHS = 40
LEARNING_RATE = 1e-3
HIDDEN_DIM = 32
SWARM_STEPS = 3

/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Исходная модель

Фиксируем версию ThermoMPNN и веса. Для каждого остатка используем
два последних слоя декодера и WT-эмбеддинг: **L × 384**.
Кэш зависит от исходников, конфигурации, весов и конкретной PDB.

In [2]:
torch.set_num_threads(min(4, os.cpu_count() or 1))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [3]:
EXPECTED_COMMIT = '370f76ec62bd929f7425e311d8df04a0d094990f'
actual_commit = subprocess.check_output(['git', '-C', str(THERMOMPNN_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == EXPECTED_COMMIT, f'Версия ThermoMPNN отличается: {actual_commit}'
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
assert checkpoint_sha == 'af449118ccfb4e34971d802321907ed82a8a035ae80f055bf8fc56009a4a838a'
if str(THERMOMPNN_DIR) not in sys.path:
    sys.path.insert(0, str(THERMOMPNN_DIR))
from train_thermompnn import TransferModelPL
from protein_mpnn_utils import parse_PDB, tied_featurize
from transfer_model import ALPHABET
assert ALPHABET == 'ACDEFGHIKLMNPQRSTVWYX'
cfg = OmegaConf.load(THERMOMPNN_DIR / 'config.yaml')
cfg.platform = {'thermompnn_dir': str(THERMOMPNN_DIR), 'accel': DEVICE.type}
reference = TransferModelPL.load_from_checkpoint(str(CHECKPOINT_PATH), cfg=cfg, map_location=DEVICE, strict=True)
reference.to(DEVICE).eval()
reference.requires_grad_(False)
core = reference.model
assert core.num_final_layers == 2
assert core.subtract_mut
feature_spec = {'format_version': 1, 'commit': actual_commit, 'checkpoint_sha256': checkpoint_sha, 'proteinmpnn_sha256': sha256_file(THERMOMPNN_DIR / 'vanilla_model_weights' / 'v_48_020.pt'), 'source_sha256': {name: sha256_file(THERMOMPNN_DIR / name) for name in ['protein_mpnn_utils.py', 'transfer_model.py']}, 'model_config': OmegaConf.to_container(cfg.model, resolve=True), 'torch': str(torch.__version__), 'device': str(DEVICE)}
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(feature_spec, sort_keys=True).encode()).hexdigest()

/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary obj

In [4]:
def pdb_path_for(name):
    stem = name.split('.pdb')[0].replace('|', ':')
    return PDB_DIR / f'{stem}.pdb'

@torch.no_grad()
def extract_features(pdb_batch):
    reference.eval()
    packed = tied_featurize([pdb_batch[0]], DEVICE, None, ca_only=False)
    X, S, mask, _, chain_M, chains = packed[:6]
    residue_idx = packed[12]
    hidden, sequence_embedding, _ = core.prot_mpnn(X, S, mask, chain_M, residue_idx, chains, None)
    features = torch.cat([*hidden[:core.num_final_layers], sequence_embedding], dim=-1)
    return (features[0].cpu(), mask[0].bool().cpu(), S[0].cpu())

@torch.no_grad()
def original_scores(features, valid):
    reference.eval()
    values = features.to(DEVICE)
    valid = valid.to(DEVICE)
    if core.lightattn:
        values = core.light_attention(values[:, :, None], valid[:, None]).reshape(len(features), -1)
    scores = core.ddg_out(core.both_out(values).unsqueeze(-1)).squeeze(-1)
    return scores.cpu()

def load_features(name, pdb_batch):
    path = pdb_path_for(name)
    sequence = pdb_batch[0]['seq']
    metadata = {'feature_signature': FEATURE_SIGNATURE, 'pdb_sha256': sha256_file(path), 'sequence': sequence, 'name': name}
    cache_key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()
    cache_path = CACHE_DIR / f'{cache_key}.pt'
    if cache_path.exists():
        saved = torch.load(cache_path, map_location='cpu', weights_only=True)
        assert saved['metadata'] == metadata
    else:
        features, valid, encoded_sequence = extract_features(pdb_batch)
        saved = {'metadata': metadata, 'features': features, 'valid': valid, 'encoded_sequence': encoded_sequence, 'baseline_scores': original_scores(features, valid)}
        torch.save(saved, cache_path)
    features = saved['features']
    valid = saved['valid']
    assert features.shape == (len(sequence), 384)
    assert valid.all(), f'{name}: неполные координаты'
    assert torch.isfinite(features).all()
    assert torch.isfinite(saved['baseline_scores']).all()
    assert not features.requires_grad
    decoded = ''.join((ALPHABET[int(i)] for i in saved['encoded_sequence']))
    assert decoded == sequence
    return saved

## Данные

Оставляем одиночные замены с конечной меткой (проверяем последовательность
WT, позиции, последовательности мутантов и полноту координат).

In [5]:
with SPLITS_PATH.open("rb") as stream:
    splits = pickle.load(stream)
train_names, val_names, test_names = [sorted(splits[key]) for key in ("train", "val", "test")]
assert tuple(map(len, (train_names, val_names, test_names))) == (239, 31, 28)
assert set(train_names).isdisjoint(val_names)
assert set(train_names).isdisjoint(test_names)
assert set(val_names).isdisjoint(test_names)

def prepare_mutations(part, sequence):
    wt_sequences = part.loc[part['mut_type'].eq('wt'), 'aa_seq'].unique()
    assert len(wt_sequences) == 1, 'Неоднозначная WT-последовательность'
    assert wt_sequences[0] == sequence, 'WT-последовательность не совпала с PDB'
    pieces = part['mut_type'].str.extract('^([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)([ACDEFGHIKLMNPQRSTVWY])$')
    target = -pd.to_numeric(part['ddG_ML'], errors='coerce')
    keep = pieces[0].notna() & pieces[0].ne(pieces[2]) & np.isfinite(target)
    rows = part.loc[keep, ['WT_name', 'mut_type', 'aa_seq']].copy()
    rows['index0'] = pieces.loc[keep, 1].astype(int) - 1
    rows['wt_aa'] = pieces.loc[keep, 0]
    rows['mut_aa'] = pieces.loc[keep, 2]
    rows['ddg_exp'] = target.loc[keep].astype(float)
    assert len(rows) > 0, 'Нет подходящих одиночных мутаций'
    for row in rows.itertuples():
        i = row.index0
        assert 0 <= i < len(sequence)
        assert sequence[i] == row.wt_aa
        assert row.aa_seq == sequence[:i] + row.mut_aa + sequence[i + 1:]
    rows['wt_index'] = rows['wt_aa'].map(ALPHABET.index)
    rows['mut_index'] = rows['mut_aa'].map(ALPHABET.index)
    return rows.drop(columns='aa_seq').reset_index(drop=True)

In [6]:
def load_records(names):
    if not CSV_PATH.exists():
        with ZipFile(DATA_DIR / "Processed_K50_dG_datasets.zip") as archive:
            archive.extract(CSV_PATH.relative_to(DATA_DIR).as_posix(), DATA_DIR)
    missing = [pdb_path_for(name) for name in names if not pdb_path_for(name).exists()]
    if missing:
        PDB_DIR.mkdir(parents=True, exist_ok=True)
        with ZipFile(DATA_DIR / "AlphaFold_model_PDBs.zip") as archive:
            members = archive.namelist()
            for path in missing:
                matches = [member for member in members if Path(member).name == path.name]
                assert len(matches) == 1, f"Не найдена однозначная структура: {path.name}"
                path.write_bytes(archive.read(matches[0]))

    with pd.read_csv(
        CSV_PATH, usecols=["WT_name", "mut_type", "aa_seq", "ddG_ML"],
        dtype=str, keep_default_na=False, chunksize=50_000,
    ) as reader:
        raw = pd.concat((chunk.loc[chunk["WT_name"].isin(names)] for chunk in reader))
    groups = raw.groupby("WT_name", sort=False)
    assert set(groups.groups) == set(names), "В CSV отсутствуют нужные белки."
    prepared, report = {}, []
    for name in tqdm(names, desc="Белки"):
        part = groups.get_group(name).reset_index(drop=True)
        pdb_batch = parse_PDB(str(pdb_path_for(name)))
        assert len(pdb_batch) == 1 and pdb_batch[0]["num_of_chains"] == 1, name
        sequence = pdb_batch[0]["seq"]
        rows = prepare_mutations(part, sequence)
        saved = load_features(name, pdb_batch)
        prepared[name] = {
            "features": saved["features"], "baseline_scores": saved["baseline_scores"],
            "rows": rows,
            "pos": torch.tensor(rows["index0"].to_numpy(), dtype=torch.long),
            "wt": torch.tensor(rows["wt_index"].to_numpy(), dtype=torch.long),
            "mut": torch.tensor(rows["mut_index"].to_numpy(), dtype=torch.long),
            "target": torch.tensor(rows["ddg_exp"].to_numpy(), dtype=torch.float32),
        }
        report.append({"protein": name, "length": len(sequence),
                       "raw_rows": len(part), "single_mutations": len(rows)})
    return prepared, pd.DataFrame(report)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
records, data_report = load_records(train_names + val_names)
data_report["split"] = np.where(data_report["protein"].isin(train_names), "train", "val")

Белки: 100%|██████████| 270/270 [00:49<00:00,  5.49it/s]


## Обучаемые головы

`mlp` обрабатывает каждый остаток независимо. Рекуррентные головы используют
одни и те же веса и три обновления: `self` получает собственное состояние,
`static` — среднее после первого шага, `swarm` — пересчитываемое среднее.
Static-сводка сохраняет градиент. Выход — 21 оценка на остаток;
**ΔΔG = оценка MUT − оценка WT**.

In [7]:
class MLPHead(nn.Module):

    def __init__(self, input_dim=384, hidden_dim=32):
        super().__init__()
        self.project = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())
        self.readout = nn.Sequential(nn.Linear(hidden_dim, 256), nn.Tanh(), nn.Linear(256, 21))

    def forward(self, features):
        return self.readout(self.project(features))

class RecurrentHead(nn.Module):

    def __init__(self, input_dim=384, hidden_dim=32, steps=3, context='mean'):
        super().__init__()
        assert context in {'mean', 'self', 'static'}
        assert steps >= 1
        self.hidden_dim = hidden_dim
        self.steps = steps
        self.context = context
        self.project = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())
        self.from_x = nn.Linear(hidden_dim, 4 * hidden_dim, bias=True)
        self.from_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)
        self.from_context = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)
        self.readout = nn.Linear(2 * hidden_dim, 21)

    def forward(self, features):
        assert features.ndim == 2 and len(features) > 0
        x = self.project(features)
        x_term = self.from_x(x)
        h = x.new_zeros(len(x), self.hidden_dim)
        c = torch.zeros_like(h)
        static_pool = None
        for step in range(self.steps):
            gates = x_term
            if step > 0:
                if self.context == 'self':
                    pooled = h
                elif self.context == 'mean':
                    pooled = h.mean(dim=0, keepdim=True)
                else:
                    if static_pool is None:
                        static_pool = h.mean(dim=0, keepdim=True)
                    pooled = static_pool
                gates = gates + self.from_h(h) + self.from_context(pooled)
            i, f, o, g = gates.chunk(4, dim=-1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
        return self.readout(torch.cat([h, c], dim=-1))

def build_head(kind):
    if kind == 'mlp':
        return MLPHead(hidden_dim=HIDDEN_DIM)
    context = {'self': 'self', 'swarm': 'mean', 'static': 'static'}[kind]
    return RecurrentHead(hidden_dim=HIDDEN_DIM, steps=SWARM_STEPS, context=context)

## Обучение и validation

Один шаг Adam использует один белок и MSE по его мутациям.
Каждый train-белок проходит один раз за эпоху. При одинаковом seed
рекуррентные варианты имеют одинаковые начальные веса и порядок белков.
Сохраняем веса лучшей эпохи по validation macro MAE.

In [8]:
def mutation_predictions(scores, record):
    device = scores.device
    pos, wt, mut = [record[key].to(device) for key in ("pos", "wt", "mut")]
    return scores[pos, mut] - scores[pos, wt]

@torch.no_grad()
def evaluate(head, names, prepared):
    if head is not None:
        head.eval()
    parts = []
    for name in names:
        record = prepared[name]
        scores = record["baseline_scores"] if head is None else head(record["features"].to(DEVICE))
        predicted = mutation_predictions(scores, record).cpu().numpy()
        assert np.isfinite(predicted).all(), f"Нечисловой прогноз: {name}"
        result = record["rows"][["WT_name", "mut_type", "index0", "ddg_exp"]].copy()
        result["ddg_pred"] = predicted
        result["error"] = result["ddg_pred"] - result["ddg_exp"]
        result["abs_error"] = result["error"].abs()
        parts.append(result)
    predictions = pd.concat(parts, ignore_index=True)
    per_protein = predictions.groupby("WT_name")["abs_error"].mean()
    metrics = {
        "macro_MAE": float(per_protein.mean()),
        "pooled_MAE": float(predictions["abs_error"].mean()),
        "pooled_RMSE": float(np.sqrt(np.mean(predictions["error"] ** 2))),
    }
    return metrics, predictions

def protein_metrics(predictions, kind, seed):
    return (predictions.groupby("WT_name", as_index=False)
            .agg(MAE=("abs_error", "mean"), mutations=("abs_error", "size"))
            .assign(model=kind, seed=seed))

In [9]:
def train_head(kind, seed):
    set_seed(seed)
    head = build_head(kind).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=LEARNING_RATE)
    rng = np.random.default_rng(seed)
    best_mae = float('inf')
    best_state = None
    best_epoch = None
    history = []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        head.train()
        losses = []
        for name in rng.permutation(train_names):
            record = records[name]
            optimizer.zero_grad(set_to_none=True)
            scores = head(record['features'].to(DEVICE))
            predicted = mutation_predictions(scores, record)
            loss = nn.functional.mse_loss(predicted, record['target'].to(DEVICE))
            assert torch.isfinite(loss), f'Нечисловая ошибка: {name}'
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        metrics, _ = evaluate(head, val_names, records)
        history.append({'epoch': epoch, 'train_MSE': float(np.mean(losses)), **metrics})
        if metrics['macro_MAE'] < best_mae:
            best_mae = metrics['macro_MAE']
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
    head.load_state_dict(best_state)
    head.eval()
    elapsed = time.perf_counter() - started
    return (head, pd.DataFrame(history), best_epoch, elapsed)

In [10]:
manifest = {
    "seeds": SEEDS, "models": MODEL_KINDS, "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE, "hidden_dim": HIDDEN_DIM, "swarm_steps": SWARM_STEPS,
    "train_names": train_names, "val_names": val_names,
    "device": str(DEVICE), "torch": str(torch.__version__),
    "feature_spec": feature_spec, "feature_signature": FEATURE_SIGNATURE,
    "target": "-ddG_ML; positive = destabilizing; kcal/mol",
    "training_loss": "MSE per protein; one protein per optimizer step",
    "selection_metric": "validation macro MAE",
    "csv_sha256": sha256_file(CSV_PATH), "splits_sha256": sha256_file(SPLITS_PATH),
}
manifest_path = RESULTS_DIR / "manifest.json"
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        "Протокол изменился: выберите другое RUN_NAME."
    )
else:
    assert not any(WEIGHTS_DIR.glob("*.pt")), "Веса без manifest: выберите другое RUN_NAME."
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
data_report.to_csv(RESULTS_DIR / "data_report.csv", index=False)

def load_head(kind, seed):
    checkpoint = torch.load(WEIGHTS_DIR / f"{kind}_seed{seed}.pt", map_location="cpu", weights_only=True)
    expected = {"kind": kind, "seed": seed, "input_dim": 384,
                "hidden_dim": HIDDEN_DIM, "steps": SWARM_STEPS,
                "feature_signature": FEATURE_SIGNATURE}
    assert all(checkpoint[key] == value for key, value in expected.items()), "Несовместимые веса."
    head = build_head(kind).to(DEVICE)
    head.load_state_dict(checkpoint["state_dict"], strict=True)
    return head.eval(), checkpoint

In [11]:
validation_rows, validation_proteins = [], []
metrics, predictions = evaluate(None, val_names, records)
predictions.to_csv(RESULTS_DIR / "thermompnn_val_predictions.csv", index=False)
validation_rows.append({"model": "ThermoMPNN", "seed": None, "best_epoch": None,
                        "trainable_parameters": 0, "training_seconds": 0.0, **metrics})
validation_proteins.append(protein_metrics(predictions, "ThermoMPNN", None))

for kind, seed in tqdm([(kind, seed) for kind in MODEL_KINDS for seed in SEEDS], desc="Головы"):
    name = f"{kind}_seed{seed}"
    path = WEIGHTS_DIR / f"{name}.pt"
    if path.exists():
        head, checkpoint = load_head(kind, seed)
    else:
        head, history, best_epoch, elapsed = train_head(kind, seed)
        checkpoint = {
            "state_dict": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "kind": kind, "seed": seed, "best_epoch": best_epoch,
            "input_dim": 384, "hidden_dim": HIDDEN_DIM, "steps": SWARM_STEPS,
            "feature_signature": FEATURE_SIGNATURE, "training_seconds": elapsed,
        }
        history.to_csv(RESULTS_DIR / f"{name}_history.csv", index=False)
        torch.save(checkpoint, path.with_suffix(".tmp"))
        path.with_suffix(".tmp").replace(path)
    metrics, predictions = evaluate(head, val_names, records)
    predictions.to_csv(RESULTS_DIR / f"{name}_val_predictions.csv", index=False)
    validation_rows.append({"model": kind, "seed": seed, "best_epoch": checkpoint["best_epoch"],
                            "trainable_parameters": sum(p.numel() for p in head.parameters()),
                            "training_seconds": checkpoint.get("training_seconds", np.nan), **metrics})
    validation_proteins.append(protein_metrics(predictions, kind, seed))
    pd.DataFrame(validation_rows).to_csv(RESULTS_DIR / "summary.csv", index=False)
    pd.concat(validation_proteins, ignore_index=True).to_csv(RESULTS_DIR / "per_protein_metrics.csv", index=False)
    del head

validation_summary = pd.DataFrame(validation_rows)
assert len(validation_summary) == 1 + len(MODEL_KINDS) * len(SEEDS)

Головы: 100%|██████████| 40/40 [24:00<00:00, 36.00s/it]


## Test

Оцениваем сохранённые validation-checkpoints для всех десяти seed.

In [12]:
TEST_RESULTS_DIR.mkdir(exist_ok=True)
test_protocol = {
    "training_manifest": manifest, "test_names": test_names,
    "primary_comparison": "MAE(static) - MAE(swarm)",
    "evaluation": "all seeds separately; validation-selected checkpoints; no ensemble",
    "weights_sha256": {f"{kind}_seed{seed}.pt": sha256_file(WEIGHTS_DIR / f"{kind}_seed{seed}.pt")
                       for kind in MODEL_KINDS for seed in SEEDS},
}
(TEST_RESULTS_DIR / "protocol.json").write_text(
    json.dumps(test_protocol, indent=2, ensure_ascii=False), encoding="utf-8"
)
test_records, test_report = load_records(test_names)
test_report.to_csv(TEST_RESULTS_DIR / "data_report.csv", index=False)

Белки: 100%|██████████| 28/28 [00:06<00:00,  4.35it/s]


In [13]:
test_rows, test_proteins = [], []
for kind, seed in [("ThermoMPNN", None), *[(kind, seed) for kind in MODEL_KINDS for seed in SEEDS]]:
    head = None if seed is None else load_head(kind, seed)[0]
    metrics, predictions = evaluate(head, test_names, test_records)
    name = "thermompnn" if seed is None else f"{kind}_seed{seed}"
    predictions.to_csv(TEST_RESULTS_DIR / f"{name}_predictions.csv", index=False)
    test_rows.append({"model": kind, "seed": seed, **metrics})
    test_proteins.append(protein_metrics(predictions, kind, seed))
    del head
test_summary = pd.DataFrame(test_rows)
test_per_protein = pd.concat(test_proteins, ignore_index=True)
assert len(test_summary) == 1 + len(MODEL_KINDS) * len(SEEDS)
test_summary.to_csv(TEST_RESULTS_DIR / "summary.csv", index=False)
test_per_protein.to_csv(TEST_RESULTS_DIR / "per_protein_metrics.csv", index=False)

## Итоговые метрики и сравнения

Macro MAE — среднее MAE по белкам; pooled MAE/RMSE — ошибки по всем мутациям.
Показываем средние метрики и разброс между seed.
В парных сравнениях положительная разница означает преимущество второй модели.

In [14]:
COMPARISONS = {
    "ThermoMPNN_minus_swarm": ("ThermoMPNN", "swarm"),
    "self_minus_static": ("self", "static"),
    "static_minus_swarm": ("static", "swarm"),
    "self_minus_swarm": ("self", "swarm"),
}

for label, table, directory in [
    ("Validation", validation_summary, RESULTS_DIR),
    ("Test", test_summary, TEST_RESULTS_DIR),
]:
    averages = table.groupby("model").agg(
        runs=("macro_MAE", "count"), macro_MAE_mean=("macro_MAE", "mean"),
        macro_MAE_std=("macro_MAE", "std"), pooled_MAE_mean=("pooled_MAE", "mean"),
        pooled_RMSE_mean=("pooled_RMSE", "mean"),
    ).reindex(["ThermoMPNN", *MODEL_KINDS])
    paired = table.loc[table["model"].isin(MODEL_KINDS)].pivot(
        index="seed", columns="model", values="macro_MAE"
    )
    paired.index = paired.index.astype(int)
    assert set(paired.index) == set(SEEDS) and not paired.isna().any().any()
    effects = []
    for name, (left, right) in COMPARISONS.items():
        if left == "ThermoMPNN":
            continue
        gain = paired[left] - paired[right]
        paired[name] = gain
        effects.append({"comparison": name, "mean_gain": gain.mean(), "std_gain": gain.std(),
                        "seeds_second_better": int((gain > 1e-6).sum()),
                        "seeds_first_better": int((gain < -1e-6).sum()),
                        "seeds_nearly_equal": int((gain.abs() <= 1e-6).sum())})
    averages.to_csv(directory / "model_averages.csv")
    paired.to_csv(directory / "paired_comparisons.csv")
    pd.DataFrame(effects).set_index("comparison").to_csv(directory / "effects_across_seeds.csv")
    print(label)
    display(averages.round(6))

Validation


,runs,macro_MAE_mean,macro_MAE_std,pooled_MAE_mean,pooled_RMSE_mean
model,,,,,
ThermoMPNN,1,0.430518,NaN,0.428947,0.583080
mlp,10,0.411275,0.002213,0.416098,0.561053
self,10,0.404478,0.002089,0.408425,0.554381
static,10,0.403343,0.001880,0.406617,0.552629
swarm,10,0.403562,0.001880,0.407470,0.554083


Test


,runs,macro_MAE_mean,macro_MAE_std,pooled_MAE_mean,pooled_RMSE_mean
model,,,,,
ThermoMPNN,1,0.529062,NaN,0.534775,0.717473
mlp,10,0.524161,0.005092,0.528500,0.713910
self,10,0.515525,0.002670,0.520109,0.710171
static,10,0.514592,0.004307,0.519826,0.707061
swarm,10,0.513064,0.005217,0.518153,0.706672


## Парный bootstrap по test-белкам

Сначала усредняем MAE каждой модели по seed внутри белка, затем
перевыбираем 28 белков с возвращением: 10 000 повторов, RNG 2026.
Все пары используют одинаковые выборки; интервал — процентили 2.5% и 97.5%
(интервал оценивает разброс по белкам при фиксированных обученных моделях.
Если он включает ноль, уверенное направление преимущества не установлено).

In [15]:
assert not test_per_protein.duplicated(["WT_name", "model", "seed"]).any()
for kind in MODEL_KINDS:
    panel = test_per_protein.loc[test_per_protein["model"] == kind].pivot(
        index="WT_name", columns="seed", values="MAE"
    )
    assert set(panel.index) == set(test_names) and set(panel.columns) == set(SEEDS)
    assert not panel.isna().any().any()
baseline = test_per_protein.loc[test_per_protein["model"] == "ThermoMPNN"]
assert len(baseline) == len(test_names) and set(baseline["WT_name"]) == set(test_names)

final_mae = test_per_protein.groupby(["WT_name", "model"])["MAE"].mean().unstack("model")
final_proteins = final_mae[["static", "swarm"]].copy()
final_proteins["gain"] = final_proteins["static"] - final_proteins["swarm"]
final_proteins.to_csv(TEST_RESULTS_DIR / "final_protein_comparison.csv")
gain = final_proteins["gain"]
distribution = pd.Series({"Белков": len(gain), "SWARM лучше": int((gain > 1e-6).sum()),
                          "Static лучше": int((gain < -1e-6).sum()),
                          "Численно совпали": int((gain.abs() <= 1e-6).sum()),
                          "Средний выигрыш": gain.mean(), "Медианный выигрыш": gain.median()})
distribution.to_csv(TEST_RESULTS_DIR / "final_distribution.csv", header=["Значение"])

indices = np.random.default_rng(2026).integers(0, len(final_mae), size=(10_000, len(final_mae)))
interval_rows = []
for name, (left, right) in COMPARISONS.items():
    differences = (final_mae[left] - final_mae[right]).to_numpy()
    low, high = np.quantile(differences[indices].mean(axis=1), [0.025, 0.975])
    interval_rows.append({"comparison": name, "mean_gain": differences.mean(),
                          "CI95_low": low, "CI95_high": high,
                          "includes_zero": bool(low <= 0 <= high)})
intervals = pd.DataFrame(interval_rows).set_index("comparison")
intervals.to_csv(TEST_RESULTS_DIR / "final_bootstrap_intervals.csv")
display(distribution.to_frame("Значение").round(6))
display(intervals.round(6))

,Значение
Белков,28.000000
SWARM лучше,15.000000
Static лучше,13.000000
Численно совпали,0.000000
Средний выигрыш,0.001528
Медианный выигрыш,0.001033


,mean_gain,CI95_low,CI95_high,includes_zero
comparison,,,,
ThermoMPNN_minus_swarm,0.015998,-0.011507,0.043193,True
self_minus_static,0.000933,-0.005867,0.007352,True
static_minus_swarm,0.001528,-0.001770,0.004901,True
self_minus_swarm,0.002461,-0.004006,0.009279,True
